# Logistic Regression Experiments

This notebook documents all logistic regression experiments on the fraud detection dataset.
Each experiment is logged as a separate section with its own cells, showing the progression
of improvements and the reasoning behind each change.

Each experiment is run on **three feature sets** to compare the effect of feature selection:

| Label | Source | Features |
|-------|--------|----------|
| **Partially Selected (PS)** | `data/processed/partially_selected_features.csv` | 33 features (all after column cleanup) |
| **Selected (S)** | `data/processed/selected_features.csv` | 20 features (MRMR-selected subset) |

**Split strategy:** 70% train / 15% validation / 15% test (stratified)

**Why stratified split?** The dataset is imbalanced (~75% non-fraud, ~25% fraud). Stratification
ensures each split preserves this ratio, preventing a split where one set has disproportionate
class distribution which would give misleading metrics.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, f1_score
)
import warnings
warnings.filterwarnings('ignore')

from src.data.load_data import load_selected_features
from src.data.preprocess import encode_target, encode_categoricals, scale_features, split_data
from src.models.predict import predict, predict_proba
from src.evaluation.metrics import evaluate_model, print_metrics

print('Imports loaded successfully.')

## Data Preparation

We load the MRMR-selected features from the feature selection step. The preprocessing pipeline:
1. **Label encoding** for categorical features — converts string categories to integers so logistic regression can process them
2. **Stratified train/val/test split** — 70/15/15 ratio, preserving class balance in each set
3. **Standard scaling** — fit on training set only, then transform val/test to prevent data leakage

**Why scale?** Logistic regression uses gradient-based optimization (L-BFGS). Features on different scales
(e.g., `total_claim_amount` in thousands vs `witnesses` in 0-5) cause the optimizer to take inefficient
zigzag paths. Scaling ensures all features contribute proportionally and regularization (C parameter)
penalizes coefficients fairly.

In [ ]:
# Prepare all datasets
def prepare_dataset(path, label):
    df = load_selected_features(path)
    print(f"\n{'='*60}")
    print(f"{label} — {path}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"Fraud rate: {(df['fraud_reported'] == 'Y').mean()*100:.1f}%")

    X = df.drop(columns=['fraud_reported'])
    y = encode_target(df['fraud_reported'])
    X_encoded, encoders = encode_categoricals(X)
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    print(f"Categorical: {len(cat_cols)}  |  Numerical: {len(num_cols)}")

    X_train, X_val, X_test, y_train, y_val, y_test = split_data(
        X_encoded, y, test_size=0.15, val_size=0.15, random_state=42
    )
    X_train_s, X_val_s, X_test_s, scaler = scale_features(X_train, X_val, X_test)
    print(f"Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_test.shape[0]}")

    return {
        'X_train': X_train_s, 'X_val': X_val_s, 'X_test': X_test_s,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
        'scaler': scaler, 'label': label,
    }

ds_ps = prepare_dataset('../data/processed/partially_selected_features.csv', 'Partially Selected (33 feat.)')
ds_s  = prepare_dataset('../data/processed/selected_features.csv', 'Selected / MRMR (20 feat.)')

datasets = [ds_ps, ds_s]

---
## Run 1: Baseline Logistic Regression

**Goal:** Establish a baseline with default logistic regression parameters.

**Configuration:**
- `C=1.0` — default regularization strength (inverse of lambda). C=1 is a moderate regularization.
- `class_weight='balanced'` — automatically adjusts weights inversely proportional to class frequencies.
  Without this, the model would optimize for the majority class (non-fraud) and predict almost everything
  as non-fraud, achieving ~75% accuracy but missing most fraud cases.
- `solver='lbfgs'` — Limited-memory Broyden-Fletcher-Goldfarb-Shanno, a quasi-Newton method good for
  small-to-medium datasets. Efficient and handles L2 regularization natively.
- `max_iter=1000` — ensures convergence (default 100 can sometimes be insufficient).

**Why balanced class weights?** In fraud detection, a false negative (missing a fraud case) is typically
more costly than a false positive (flagging a legitimate claim). Balanced weights tell the model that
misclassifying a fraud case is ~3x worse than misclassifying a non-fraud case (proportional to 750/250).

In [ ]:
# Run 1: Baseline on all feature sets
for ds in datasets:
    model = LogisticRegression(
        C=1.0, l1_ratio=0, solver='lbfgs', max_iter=1000,
        class_weight='balanced', random_state=42
    )
    model.fit(ds['X_train'], ds['y_train'])
    ds['model_v1'] = model
    ds['results_log'] = []

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    for name, X_set, y_set in [("Train", ds['X_train'], ds['y_train']),
                                ("Validation", ds['X_val'], ds['y_val'])]:
        y_pred = predict(model, X_set)
        y_prob = predict_proba(model, X_set)
        res = evaluate_model(y_set, y_pred, y_prob)
        print_metrics(res, name)
        print()
        if name == "Validation":
            ds['val_res_v1'] = res

    ds['results_log'].append({
        'Run': 'V1: Baseline (C=1.0, balanced)',
        'Val F1': ds['val_res_v1']['f1'],
        'Val Precision': ds['val_res_v1']['precision'],
        'Val Recall': ds['val_res_v1']['recall'],
        'Val ROC AUC': ds['val_res_v1']['roc_auc'],
    })

In [ ]:
# Confusion matrices — baseline
fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 4))
for ax, ds in zip(axes, datasets):
    y_pred = predict(ds['model_v1'], ds['X_val'])
    cm = confusion_matrix(ds['y_val'], y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    ax.set_title(f'V1 — {ds["label"][:20]}')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

---
## Run 2: Cross-Validation to Assess Stability

**Goal:** Before tuning hyperparameters, assess how stable the baseline model is across different
data folds. This tells us if the validation score is reliable or just lucky.

**Why cross-validation here?** With only 700 training samples, a single train/val split can be noisy.
5-fold CV gives us 5 different validation estimates, and the variance tells us how sensitive the
model is to which data ends up in train vs validation.

**Method:** 5-fold stratified CV on the training set. We use stratified folds to maintain class
balance in each fold, and score with F1 since accuracy is misleading on imbalanced data.

**Note:** We run CV on training data only. The test set remains untouched until final evaluation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for ds in datasets:
    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    cv_model = LogisticRegression(C=1.0, l1_ratio=0, solver='lbfgs', max_iter=1000,
                                   class_weight='balanced', random_state=42)
    for metric in ['f1', 'precision', 'recall', 'roc_auc']:
        scores = cross_val_score(cv_model, ds['X_train'], ds['y_train'], cv=cv, scoring=metric)
        print(f"{metric:<12}: {scores.mean():.4f} (+/- {scores.std():.4f})")

---
## Run 3: Regularization Tuning (C parameter)

**Goal:** Find the optimal regularization strength.

**Why tune C?** The C parameter controls the trade-off between fitting the training data and keeping
coefficients small (generalization):
- **Low C** (e.g., 0.01) = strong regularization = simpler model, may underfit
- **High C** (e.g., 100) = weak regularization = complex model, may overfit
- We search across a log-scale range to find the sweet spot

**Method:** Evaluate C values on the validation set. We use the validation set (not CV here) because
we want a quick comparison across many C values. The best C will later be validated with CV.

In [ ]:
C_values = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]

for ds in datasets:
    c_results = []
    for C in C_values:
        model_c = LogisticRegression(C=C, l1_ratio=0, solver='lbfgs', max_iter=1000,
                                      class_weight='balanced', random_state=42)
        model_c.fit(ds['X_train'], ds['y_train'])
        y_pred_val = predict(model_c, ds['X_val'])
        y_prob_val = predict_proba(model_c, ds['X_val'])
        val_res = evaluate_model(ds['y_val'], y_pred_val, y_prob_val)
        train_f1 = f1_score(ds['y_train'], predict(model_c, ds['X_train']))
        c_results.append({'C': C, 'Train F1': train_f1, 'Val F1': val_res['f1'],
                          'Val Precision': val_res['precision'], 'Val Recall': val_res['recall'],
                          'Val ROC AUC': val_res['roc_auc']})

    c_df = pd.DataFrame(c_results)
    ds['c_df'] = c_df
    ds['best_c'] = c_df.loc[c_df['Val F1'].idxmax(), 'C']

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(c_df.to_string(index=False))
    print(f"\nBest C: {ds['best_c']}")

In [ ]:
fig, axes = plt.subplots(len(datasets), 2, figsize=(14, 5*len(datasets)))
for row, ds in enumerate(datasets):
    c_df = ds['c_df']
    axes[row, 0].semilogx(c_df['C'], c_df['Train F1'], 'o-', label='Train F1')
    axes[row, 0].semilogx(c_df['C'], c_df['Val F1'], 's-', label='Val F1')
    axes[row, 0].axvline(x=ds['best_c'], color='red', linestyle='--', alpha=0.5, label=f'Best C={ds["best_c"]}')
    axes[row, 0].set_xlabel('C'); axes[row, 0].set_ylabel('F1'); axes[row, 0].set_title(f'F1 vs C — {ds["label"]}')
    axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].semilogx(c_df['C'], c_df['Val Precision'], 'o-', label='Precision')
    axes[row, 1].semilogx(c_df['C'], c_df['Val Recall'], 's-', label='Recall')
    axes[row, 1].axvline(x=ds['best_c'], color='red', linestyle='--', alpha=0.5, label=f'Best C={ds["best_c"]}')
    axes[row, 1].set_xlabel('C'); axes[row, 1].set_ylabel('Score'); axes[row, 1].set_title(f'P/R vs C — {ds["label"]}')
    axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Run 4: Best C with Cross-Validation Confirmation

**Goal:** Confirm the best C found in Run 3 is genuinely better, not just lucky on this particular
validation split.

**Method:** Run 5-fold stratified CV with the best C on the full training+validation data to get
a robust performance estimate. Then retrain on full train set and evaluate on validation.

In [ ]:
for ds in datasets:
    best_c = ds['best_c']
    model_v2 = LogisticRegression(C=best_c, l1_ratio=0, solver='lbfgs', max_iter=1000,
                                   class_weight='balanced', random_state=42)
    cv_scores = cross_val_score(model_v2, ds['X_train'], ds['y_train'], cv=cv, scoring='f1')
    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(f"CV F1 with C={best_c}: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    model_v2.fit(ds['X_train'], ds['y_train'])
    ds['model_v2'] = model_v2

    for name, X_set, y_set in [("Train", ds['X_train'], ds['y_train']),
                                ("Validation", ds['X_val'], ds['y_val'])]:
        y_pred = predict(model_v2, X_set)
        y_prob = predict_proba(model_v2, X_set)
        res = evaluate_model(y_set, y_pred, y_prob)
        print_metrics(res, name)
        print()
        if name == "Validation":
            ds['val_res_v2'] = res

    ds['results_log'].append({
        'Run': f'V2: Tuned C={best_c}',
        'Val F1': ds['val_res_v2']['f1'], 'Val Precision': ds['val_res_v2']['precision'],
        'Val Recall': ds['val_res_v2']['recall'], 'Val ROC AUC': ds['val_res_v2']['roc_auc'],
    })

---
## Run 5: Threshold Tuning

**Goal:** The default classification threshold is 0.5, but this may not be optimal for imbalanced data.

**Why tune the threshold?** Logistic regression outputs a probability. The threshold decides where we
draw the line between "fraud" and "not fraud":
- **Lower threshold** (e.g., 0.3) = more fraud predictions = higher recall, lower precision
- **Higher threshold** (e.g., 0.7) = fewer fraud predictions = lower recall, higher precision

In fraud detection, we typically prefer higher recall (catch more fraud) even at the cost of some
precision (more false alarms), because the cost of missing fraud >> cost of investigating a false alarm.

**Method:** Sweep thresholds from 0.1 to 0.9 on the validation set and pick the one that maximizes F1.

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
for ds in datasets:
    y_prob_val = predict_proba(ds['model_v2'], ds['X_val'])
    threshold_results = []
    for t in thresholds:
        y_pred_t = (y_prob_val >= t).astype(int)
        if y_pred_t.sum() == 0 or y_pred_t.sum() == len(y_pred_t): continue
        res = evaluate_model(ds['y_val'], y_pred_t, y_prob_val)
        threshold_results.append({'Threshold': t, 'Precision': res['precision'],
                                  'Recall': res['recall'], 'F1': res['f1'], 'Accuracy': res['accuracy']})

    t_df = pd.DataFrame(threshold_results)
    ds['t_df'] = t_df
    ds['best_threshold'] = t_df.loc[t_df['F1'].idxmax(), 'Threshold']

    print(f"\n{'='*60}")
    print(f"{ds['label']}")
    print(f"{'='*60}")
    print(t_df.to_string(index=False))
    print(f"\nBest threshold: {ds['best_threshold']:.2f}")

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(6*len(datasets), 5))
for ax, ds in zip(axes, datasets):
    t_df = ds['t_df']
    ax.plot(t_df['Threshold'], t_df['Precision'], 'o-', label='Precision')
    ax.plot(t_df['Threshold'], t_df['Recall'], 's-', label='Recall')
    ax.plot(t_df['Threshold'], t_df['F1'], '^-', label='F1', linewidth=2)
    ax.axvline(x=ds['best_threshold'], color='red', linestyle='--', alpha=0.5, label=f'Best={ds["best_threshold"]:.2f}')
    ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
    ax.set_xlabel('Threshold'); ax.set_ylabel('Score'); ax.set_title(f'Threshold — {ds["label"][:20]}')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
for ds in datasets:
    bt = ds['best_threshold']
    print(f"\n{'='*60}")
    print(f"{ds['label']}  —  threshold = {bt:.2f}")
    print(f"{'='*60}")
    for name, X_set, y_set in [("Train", ds['X_train'], ds['y_train']),
                                ("Validation", ds['X_val'], ds['y_val'])]:
        y_prob = predict_proba(ds['model_v2'], X_set)
        y_pred_t = (y_prob >= bt).astype(int)
        res = evaluate_model(y_set, y_pred_t, y_prob)
        print_metrics(res, name)
        print()
        if name == "Validation":
            ds['val_res_v3'] = res

    ds['results_log'].append({
        'Run': f'V3: C={ds["best_c"]}, threshold={bt:.2f}',
        'Val F1': ds['val_res_v3']['f1'], 'Val Precision': ds['val_res_v3']['precision'],
        'Val Recall': ds['val_res_v3']['recall'], 'Val ROC AUC': ds['val_res_v3']['roc_auc'],
    })

---
## Run 6: No Class Weighting (Comparison)

**Goal:** Understand the effect of `class_weight='balanced'` by removing it.

**Why this experiment?** We need to quantify how much the balanced weighting helps. Without it,
the model treats all misclassifications equally, which on imbalanced data typically leads to:
- Higher accuracy (predicts majority class more)
- Lower recall (misses more fraud cases)
- This comparison justifies our choice of balanced weights.

In [ ]:
for ds in datasets:
    model_nw = LogisticRegression(C=ds['best_c'], l1_ratio=0, solver='lbfgs', max_iter=1000,
                                   class_weight=None, random_state=42)
    model_nw.fit(ds['X_train'], ds['y_train'])
    print(f"\n{'='*60}")
    print(f"{ds['label']} — WITHOUT class_weight='balanced'")
    print(f"{'='*60}")
    y_pred = predict(model_nw, ds['X_val'])
    y_prob = predict_proba(model_nw, ds['X_val'])
    res = evaluate_model(ds['y_val'], y_pred, y_prob)
    print_metrics(res, 'Validation')
    print()
    ds['results_log'].append({
        'Run': f'V4: No class weighting (C={ds["best_c"]})',
        'Val F1': res['f1'], 'Val Precision': res['precision'],
        'Val Recall': res['recall'], 'Val ROC AUC': res['roc_auc'],
    })

---
## Run Summary & Model Selection

Compare all experiment runs on **validation metrics only**. The test set has been held out
throughout all tuning and will be evaluated exactly once in the final section below.

In [ ]:
for ds in datasets:
    summary_df = pd.DataFrame(ds['results_log'])
    ds['summary_df'] = summary_df
    print(f"\n{'='*90}")
    print(f"EXPERIMENT LOG — {ds['label']}")
    print(f"{'='*90}")
    print(summary_df.to_string(index=False))
    best_idx = summary_df['Val F1'].idxmax()
    ds['best_run_name'] = summary_df.loc[best_idx, 'Run']
    print(f"\nBest: {ds['best_run_name']} (F1={summary_df.loc[best_idx, 'Val F1']:.4f})")

---
## Final Test Evaluation (ONE-TIME)

**This is the only place in the notebook where the test set is used.**

After all hyperparameter tuning and model selection on the validation set, we evaluate the
best configuration on the held-out test set exactly once. This gives an unbiased estimate
of how the model will perform on truly unseen data.

**Best configuration selected:** Based on validation F1 from the summary above.

In [ ]:
for ds in datasets:
    bt = ds['best_threshold']
    print(f"\n{'='*60}")
    print(f"FINAL TEST — {ds['label']}")
    print(f"{'='*60}")
    print(f"Model: LR(C={ds['best_c']}, balanced), threshold={bt:.2f}\n")
    y_prob_test = predict_proba(ds['model_v2'], ds['X_test'])
    y_pred_test = (y_prob_test >= bt).astype(int)
    test_res = evaluate_model(ds['y_test'], y_pred_test, y_prob_test)
    print_metrics(test_res, "Test")
    print("\nClassification Report:")
    print(classification_report(ds['y_test'], y_pred_test, target_names=['Non-Fraud', 'Fraud']))
    ds['test_res'] = test_res
    ds['y_prob_test'] = y_prob_test
    ds['y_pred_test'] = y_pred_test

In [ ]:
# Final visualizations
n_ds = len(datasets)
fig, axes = plt.subplots(2, max(n_ds, 2), figsize=(7*max(n_ds, 2), 10))
colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, ds in enumerate(datasets):
    fpr, tpr, _ = roc_curve(ds['y_test'], ds['y_prob_test'])
    roc_auc_val = auc(fpr, tpr)
    axes[0, 0].plot(fpr, tpr, color=colors[i], linewidth=2,
                    label=f'{ds["label"][:15]} (AUC={roc_auc_val:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0, 0].set_xlabel('FPR'); axes[0, 0].set_ylabel('TPR')
axes[0, 0].set_title('ROC Curves (Test)'); axes[0, 0].legend(fontsize=7); axes[0, 0].grid(True, alpha=0.3)

for i, ds in enumerate(datasets):
    prec, rec, _ = precision_recall_curve(ds['y_test'], ds['y_prob_test'])
    axes[0, 1].plot(rec, prec, color=colors[i], linewidth=2, label=ds['label'][:15])
axes[0, 1].set_xlabel('Recall'); axes[0, 1].set_ylabel('Precision')
axes[0, 1].set_title('PR Curves (Test)'); axes[0, 1].legend(fontsize=7); axes[0, 1].grid(True, alpha=0.3)

for j in range(2, max(n_ds, 2)):
    axes[0, j].set_visible(False)

for i, ds in enumerate(datasets):
    cm = confusion_matrix(ds['y_test'], ds['y_pred_test'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, i],
                xticklabels=['Non-Fraud', 'Fraud'], yticklabels=['Non-Fraud', 'Fraud'])
    axes[1, i].set_title(f'CM — {ds["label"][:20]}')
    axes[1, i].set_ylabel('Actual'); axes[1, i].set_xlabel('Predicted')
plt.tight_layout()
plt.show()

for ds in datasets:
    coef_df = pd.DataFrame({'Feature': ds['X_train'].columns, 'Coefficient': ds['model_v2'].coef_[0]}).sort_values('Coefficient')
    plt.figure(figsize=(10, max(6, len(coef_df) * 0.3)))
    plt.barh(range(len(coef_df)), coef_df['Coefficient'],
             color=['#e74c3c' if c < 0 else '#2ecc71' for c in coef_df['Coefficient']])
    plt.yticks(range(len(coef_df)), coef_df['Feature'], fontsize=8)
    plt.xlabel('Coefficient'); plt.title(f'Coefficients — {ds["label"]}')
    plt.axvline(x=0, color='black', linewidth=0.5); plt.tight_layout(); plt.show()